# Experiment 6 — XGBoost Combined (Class Weights + Threshold Optimization)

Best of both worlds:
- scale_pos_weight fixes gradient bias during training
- Threshold scan on validation set finds best decision boundary

This combination addresses both training and prediction imbalance simultaneously.

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

BASE_PARAMS = dict(n_estimators=500, max_depth=6, learning_rate=0.1,
                   random_state=42, eval_metric='auc', verbosity=0,
                   use_label_encoder=False)
THRESHOLDS  = np.arange(0.1, 0.91, 0.01)
print("Experiment 6 — XGBoost Combined (Class Weights + Threshold Optimization)")

Experiment 6 — XGBoost Combined (Class Weights + Threshold Optimization)


In [2]:
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp6-XGB] Training Version {v}...")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_tr_full = train.drop('label',axis=1).values; y_tr_full = train['label'].values
    X_test    = test.drop('label',axis=1).values;  y_test    = test['label'].values

    n_bg = (y_tr_full==0).sum(); n_sig = (y_tr_full==1).sum()
    spw  = n_bg / n_sig
    print(f"[Exp6-XGB] scale_pos_weight={spw:.2f}")

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr_full, y_tr_full, test_size=0.2, random_state=42, stratify=y_tr_full)

    model = XGBClassifier(**{**BASE_PARAMS, 'scale_pos_weight': spw})
    t0    = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    val_probs = model.predict_proba(X_val)[:,1]
    f1_scores = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
                 for t in THRESHOLDS]
    best_threshold = THRESHOLDS[int(np.argmax(f1_scores))]
    print(f"[Exp6-XGB] best_threshold={best_threshold:.2f} | val_F1={max(f1_scores):.4f}")

    probs = model.predict_proba(X_test)[:,1]
    preds = (probs >= best_threshold).astype(int)

    metrics = compute_all_metrics(y_test, probs, preds, train_time)
    metrics['Version']        = v
    metrics['Best_Threshold'] = round(best_threshold, 2)
    all_results.append(metrics)

    np.save(os.path.join(RESULTS_DIR, f'probs_exp6_xgb_version_{v}.npy'), probs)
    print_metrics(metrics, label=f"Exp6-XGB Version {v}")

print("\n[Exp6-XGB] Complete.")


[Exp6-XGB] Training Version A...
[Exp6-XGB] scale_pos_weight=0.89
[Exp6-XGB] best_threshold=0.37 | val_F1=0.7698
[Exp6-XGB Version A] AUC=0.8205 | F1=0.7693 | Signal_Sig=178.5115 | Train_Time=632.17s

[Exp6-XGB] Training Version B...
[Exp6-XGB] scale_pos_weight=10.00
[Exp6-XGB] best_threshold=0.64 | val_F1=0.3818
[Exp6-XGB Version B] AUC=0.8006 | F1=0.3912 | Signal_Sig=25.9188 | Train_Time=585.34s

[Exp6-XGB] Training Version C...
[Exp6-XGB] scale_pos_weight=50.01
[Exp6-XGB] best_threshold=0.58 | val_F1=0.1666
[Exp6-XGB Version C] AUC=0.7433 | F1=0.1649 | Signal_Sig=5.3129 | Train_Time=456.46s

[Exp6-XGB] Complete.


In [3]:
results_df = pd.DataFrame(all_results)[['Version','AUC','F1','Signal_Significance','Train_Time_sec','Best_Threshold']]
save_path  = os.path.join(RESULTS_DIR, 'experiment6_combined_xgb.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      AUC       F1  Signal_Significance  Train_Time_sec  Best_Threshold
      A 0.820544 0.769317           178.511479          632.17            0.37
      B 0.800586 0.391188            25.918758          585.34            0.64
      C 0.743318 0.164907             5.312915          456.46            0.58

✓ Saved → ..\results\experiment6_combined_xgb.csv
